# Lab: Stock Trading Agent — Evaluation with Langfuse

This lab builds a **stock trading assistant** with three tools and evaluates it
using Langfuse datasets and custom scoring.

## Structure

| Section | What you will do |
|---------|------------------|
| **Setup** | Run once — loads LLM, Langfuse client, and API keys |
| **Tools** | Fill in tool descriptions so the LLM knows when to use each tool |
| **Agent** | Write a system prompt and create the agent |
| **Test** | Run queries to verify the agent works correctly |
| **Evaluation** | Create a Langfuse dataset, implement custom evaluators, and run an experiment |

> **Goal:** learn how to evaluate an LLM agent using Langfuse's dataset and
> experiment features — including custom evaluators and LLM-as-judge scoring.

## Setup

1. Run the setup cell below first — all other cells depend on it.
2. Set `MODEL_PROVIDER = "openai"` or `"gemini"`.
3. Required keys in `.env`: `LANGFUSE_SECRET_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_HOST` (or `LANGFUSE_BASE_URL`), `TAVILY_API_KEY`, plus the provider key.
4. If packages are missing:
   ```bash
   uv add langfuse yfinance tavily-python langgraph
   ```

In [ ]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langfuse import Langfuse
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert LANGFUSE_SECRET_KEY, "LANGFUSE_SECRET_KEY not found — add it to your .env file"
assert LANGFUSE_PUBLIC_KEY, "LANGFUSE_PUBLIC_KEY not found — add it to your .env file"
assert TAVILY_API_KEY, "TAVILY_API_KEY not found — add it to your .env file"

# ── Set env vars for Langfuse SDK auto-detection ─────────────────────────────
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST

# ── Initialize Langfuse client ────────────────────────────────────────────────
langfuse = Langfuse(
    secret_key=LANGFUSE_SECRET_KEY,
    public_key=LANGFUSE_PUBLIC_KEY,
    host=LANGFUSE_HOST,
)

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

---
## Tools

The agent needs three tools. The implementation is provided, but the **tool descriptions
(docstrings)** are missing — you need to write them.

> **Why this matters:** the `@tool` decorator uses the docstring as the tool's description.
> The LLM reads this description to decide *when* and *how* to call each tool.
> Without a good description, the agent won't know when to use the tool or what arguments to pass.

| Tool | What it does |
|------|-------------|
| `get_price` | Fetches historical stock prices via Yahoo Finance |
| `get_news` | Searches for related news using Tavily |
| `buy_ticker` | Mock buy-order tool |

In [ ]:
@tool
def get_price(ticker: str, date_range: str) -> str:
    # TODO: Write a docstring that describes:
    #   - What the tool does 
    #   - Args: ticker?, date_range?
    #   - Returns: ?
    """TODO: write tool description here"""
    logger.info("get_price called | ticker=%s  date_range=%s", ticker, date_range)
    stock = yf.Ticker(ticker)
    hist = stock.history(period=date_range)
    if hist.empty:
        return f"Could not fetch price data for {ticker} over {date_range}."

    latest_close = hist["Close"].iloc[-1]
    period_high = hist["High"].max()
    period_low = hist["Low"].min()
    first_close = hist["Close"].iloc[0]
    change = latest_close - first_close
    pct = (change / first_close) * 100

    result = (
        f"{ticker} ({date_range}): latest close ${latest_close:.2f}, "
        f"high ${period_high:.2f}, low ${period_low:.2f}, "
        f"change {change:+.2f} ({pct:+.2f}%)"
    )
    logger.info("get_price result: %s", result)
    return result


logger.info("Tool ready: get_price")

In [ ]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def get_news(ticker: str, search_term: str) -> str:
    # TODO: Write a docstring that describes:
    #  
    """TODO: write tool description here"""
    query = f"{ticker} {search_term}"
    logger.info("get_news called | query='%s'", query)

    response = tavily_client.search(query=query, max_results=3)
    results = response.get("results", [])

    if not results:
        return f"No news found for '{query}'."

    lines = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "No title")
        snippet = r.get("content", "")[:200]
        url = r.get("url", "")
        lines.append(f"{i}. {title}\n   {snippet}\n   {url}")

    result = "\n\n".join(lines)
    logger.info("get_news returned %d results", len(results))
    return result


logger.info("Tool ready: get_news")

In [ ]:
@tool
def buy_ticker(ticker: str, amount: float, target_price: float) -> str:
    # TODO: Write a docstring that describes:
    """TODO: write tool description here"""
    logger.info(
        "buy_ticker called | ticker=%s  amount=%.2f  target_price=%.2f",
        ticker, amount, target_price,
    )
    result = (
        f"Done, a buy order has been set for {ticker} "
        f"with amount of {amount} shares at price ${target_price:.2f}"
    )
    logger.info("buy_ticker result: %s", result)
    return result


logger.info("Tool ready: buy_ticker")

---
## Build the Agent

Create a ReAct agent using `create_agent` from LangChain with:
- A **system prompt** that tells the agent its role and behavior
- The three tools defined above
- A `MemorySaver` checkpointer for multi-turn conversation memory

### Your task
1. Write a system prompt that describes the agent's role as a stock trading assistant
2. Create the agent using `create_agent`

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# TODO: Write a system prompt for the stock trading assistant.
# It should tell the agent:
# - Its role 
# - What it can do 
# - How to behave 
SYSTEM_PROMPT = ""  # TODO: fill in the system prompt

tools = [get_price, get_news, buy_ticker]
checkpointer = MemorySaver()

# TODO: Create the agent using create_agent with model, tools, system_prompt, and checkpointer


logger.info("Agent created with tools: %s", [t.name for t in tools])

---
## Test the Agent

Run a few queries to verify each tool works correctly.
We use the same `thread_id` across turns so the agent remembers context.

> **Note:** These test cells will fail if you haven't completed the **Tools** and **Build the Agent** sections above.

In [ ]:
config = {"configurable": {"thread_id": "test-session-1"}}

# ── Test 1: Price lookup ───────────────────────────────────────────────────────
result = agent.invoke(
    {"messages": [HumanMessage(content="What is the price of NVDA over the last 5 days?")]},
    config=config,
)
print("=== Price Lookup ===")
print(result["messages"][-1].content)

In [ ]:
# ── Test 2: News search ────────────────────────────────────────────────────────
result2 = agent.invoke(
    {"messages": [HumanMessage(content="Find me news about TSLA related to earnings")]},
    config=config,
)
print("=== News Search ===")
print(result2["messages"][-1].content)

In [ ]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Should I buy it")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

In [ ]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Yes")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

In [ ]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Buy 100 shares of it at $370")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

---
## Evaluation with Langfuse

Now we evaluate the agent systematically using Langfuse datasets and experiments.

We will:
1. **Create a dataset** with example queries and expected outputs using `langfuse.create_dataset()` and `langfuse.create_dataset_item()`
2. **Define evaluators** that return `Evaluation` objects — one checks tool usage, one uses LLM-as-judge
3. **Run `dataset.run_experiment()`** to evaluate the agent and view results in Langfuse

### Key Langfuse concepts

| Concept | Description |
|---------|-------------|
| **Dataset** | A named collection of test items, each with `input` and `expected_output` |
| **Dataset Item** | A single test case with input data and expected output |
| **Evaluator** | A function that scores agent output, returns `Evaluation(name, value, comment)` |
| **Experiment** | A run of the agent over all dataset items, with evaluator scores attached |

### 1. Create (or load) the Dataset

Use the Langfuse API to create a dataset and populate it with test items.

```python
# Create a dataset (idempotent — returns existing if same name)
langfuse.create_dataset(name="...", description="...")

# Add items one by one
langfuse.create_dataset_item(
    dataset_name="...",
    input={"question": "..."},
    expected_output={"expected_tool": "...", "expected_answer": "..."},
)
```

In [ ]:
DATASET_NAME = "stock-trading-agent-eval"

examples = [
    {
        "input": {"question": "What is the price of AAPL over the last month?"},
        "expected_output": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain AAPL price data including close price, high, low, and percentage change over the last month.",
        },
    },
    {
        "input": {"question": "Find news about NVDA related to AI chips"},
        "expected_output": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about NVDA related to AI chips, including titles and links.",
        },
    },
    {
        "input": {"question": "Buy 50 shares of TSLA at $250"},
        "expected_output": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for TSLA with 50 shares at $250.",
        },
    },
    {
        "input": {"question": "What is GOOGL stock price for the past 5 days?"},
        "expected_output": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain GOOGL price data over 5 days with close, high, low, and change.",
        },
    },
    {
        "input": {"question": "Search for recent AAPL news about iPhone sales"},
        "expected_output": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about AAPL and iPhone sales.",
        },
    },
    {
        "input": {"question": "Place a buy order for 100 shares of AAPL at $190"},
        "expected_output": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for AAPL with 100 shares at $190.",
        },
    },
]

# TODO: Create the dataset using langfuse.create_dataset()
# dataset = langfuse.create_dataset(
#...
# )

# TODO: Loop through examples and add each as a dataset item
# for ex in examples:
#     langfuse.create_dataset_item(
# ...
#     )

TODO: logger.info("Dataset '%s' created with %d examples", DATASET_NAME, len(examples))

### 2. Define Evaluators

Evaluators are functions that score the agent's output. In Langfuse, evaluators
return `Evaluation` objects with a name, numeric value, and optional comment.

```python
from langfuse import Evaluation

def my_evaluator(*, output, expected_output, **kwargs):
    score = ...  # your scoring logic
    return Evaluation(name="metric_name", value=score, comment="explanation")
```

You need to implement two evaluators:

| Evaluator | What it checks |
|-----------|---------------|
| `correct_tool_used` | Did the agent call the expected tool? (inspect `output["messages"]` for `tool_calls`) |
| `answer_correctness` | LLM-as-judge — does the final answer match the expected criteria? |

In [ ]:
from langfuse import Evaluation


def correct_tool_used(*, output, expected_output, **kwargs):
    """Check whether the agent called the expected tool."""
    # TODO: Implement this evaluator
    # 1. Get the expected tool name from expected_output["expected_tool"]
    # 2. Loop through output["messages"] and check each message for tool_calls:
    #      for msg in messages:
    #          if hasattr(msg, "tool_calls") and msg.tool_calls:
    #              for tc in msg.tool_calls:
    #                  if tc["name"] == expected_tool:
    #                      return Evaluation(name="correct_tool_used", value=1.0, comment=f"Called {expected_tool}")
    # 3. If the expected tool was not found:
    #      return Evaluation(name="correct_tool_used", value=0.0, comment=f"Did not call {expected_tool}")
    pass


judge_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)


def answer_correctness(*, output, expected_output, **kwargs):
    """LLM-as-judge: does the final answer satisfy the expected criteria?"""
    # TODO: Implement this evaluator
    # 1. Get the actual answer: output["messages"][-1].content
    # 2. Get the expected criteria: expected_output["expected_answer"]
    # 3. Build a judge prompt:
    #      system: "You are an evaluation judge. ... Respond ONLY with 'CORRECT' or 'INCORRECT'."
    #      user: f"ACTUAL ANSWER:\n{actual_answer}\n\nEXPECTED CRITERIA:\n{expected}"
    # 4. Call judge_llm.invoke([system_msg, user_msg])
    # 5. Check if response.content.strip().upper() == "CORRECT"
    # 6. Return Evaluation(name="answer_correctness", value=1.0 or 0.0, comment=...)
    pass


logger.info("Evaluators defined: correct_tool_used, answer_correctness")

### 3. Run Evaluation

Create a **stateless** agent (no checkpointer) so each evaluation example runs independently.
Then use `dataset.run_experiment()` to run the agent against every dataset item and score results.

```python
# The experiment runner handles:
# - Concurrent execution with max_concurrency
# - Automatic tracing of all executions
# - Running evaluators on each item's output
# - Linking traces to dataset items in the Langfuse UI

dataset = langfuse.get_dataset("dataset-name")
result = dataset.run_experiment(
    name="experiment-name",
    task=my_task_function,
    evaluators=[evaluator_1, evaluator_2],
    max_concurrency=2,
)
```

In [ ]:
eval_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)


# TODO: Define the task function that converts dataset items to agent invocations.
# The task function receives a DatasetItem via the `item` keyword argument.
# Access the question with: item.input["question"]
#
# def agent_task(*, item, **kwargs):
# 


# TODO: Load the dataset and run the experiment
# dataset = langfuse.get_dataset(DATASET_NAME)
#
# experiment_results = dataset.run_experiment(
# ...
# )

TODO: logger.info("Evaluation complete — check results in Langfuse dashboard")

In [ ]:
TODO: print(experiment_results.format())